# 1. Load and Inspect the Raw FIFA World Cup 2026 Match Data

This notebook cleans the raw FIFA World Cup 2026 match dataset.

The dataset contains 104 matches. Each match includes the round, date, home team, away team, final score, attendance, venue, referee, and match report information.

The main goals of this notebook are to:

1. Load the raw CSV file.
2. Remove blank separator rows.
3. Clean team names and match scores.
4. Remove penalty-shootout goals from knockout matches.
5. Create a clean match-level dataset with one row per match.

In [25]:
# Import the Python libraries needed for data loading, cleaning, and analysis

import pandas as pd
import numpy as np
import re

## 1.1 Load the Raw Dataset

We load the raw World Cup match data from the `data/raw` folder. This file contains one row for each of the 104 matches played during the tournament.

In [26]:
# Load the raw CSV file containing all FIFA World Cup 2026 matches

raw = pd.read_csv("../data/raw/worldcup2026_matches_raw.csv")

# Display the first five rows to check that the file has loaded correctly
raw.head()

,Round,Wk,Day,Date,Time,Home,Score,Away,Attendance,Venue,Referee,Match Report,Notes
0,Group stage,1.0,Thu,2026-06-11,13:00 (04:30),Mexico mx,2–0,za South Africa,80824.0,Estadio Banorte (Neutral Site),Wilton Sampaio,Match Report,NaN
1,Group stage,1.0,Thu,2026-06-11,20:00 (11:30),Korea Republic kr,2–1,cz Czechia,44985.0,Estadio Akron (Neutral Site),Amin Omar,Match Report,NaN
2,Group stage,1.0,Fri,2026-06-12,15:00 (04:30),Canada ca,1–1,ba Bosnia–Herz,43002.0,BMO Field (Neutral Site),Facundo Tello,Match Report,NaN
3,Group stage,1.0,Fri,2026-06-12,18:00 (10:30),USA us,4–1,py Paraguay,70492.0,SoFi Stadium (Neutral Site),Danny Makkelie,Match Report,NaN
4,Group stage,1.0,Sat,2026-06-13,12:00 (04:30),Qatar qa,1–1,ch Switzerland,67966.0,Levi's Stadium (Neutral Site),Said Martínez,Match Report,NaN


In [27]:
# Check the number of rows and columns in the raw dataset

raw.shape

(107, 13)

The raw dataset should contain 104 match rows. Blank rows between tournament rounds will be removed in the next step.

## 2. Remove Blank Separator Rows

The copied dataset contains blank rows between the group stage, knockout stage, and final matches. These rows do not contain match information and must be removed before analysis.

In [28]:
# Remove rows where the Home team column is empty
# These are blank separator rows between different tournament rounds

df = raw.dropna(subset=["Home"]).copy()
df = df[df["Home"].str.strip() != ""]

# Check how many valid matches remain after removing blank rows
print("Number of matches after removing blank rows:", len(df))

Number of matches after removing blank rows: 104


After removing blank rows, the dataset should contain exactly 104 matches.

## 3. Clean Team Names and Match Scores

Team names currently include country codes, such as `Mexico mx`. Match scores in knockout rounds also include penalty-shootout results, such as `(3) 1–1 (4)`.

We need to:

1. Remove country codes from team names.
2. Remove penalty-shootout numbers from scores.
3. Keep only the goals scored during normal time and extra time.

In [33]:
# Remove country codes from the beginning or end of team names.
# Examples:
# "Mexico mx" becomes "Mexico".
# "za South Africa" becomes "South Africa".
def clean_team_name(name):
    name = str(name).strip()
    name = re.sub(r"^[a-z]{2,3}\s+", "", name)
    name = re.sub(r"\s+[a-z]{2,3}$", "", name)
    return name.strip()


# Remove penalty-shootout numbers while keeping the match score.
# Example: "(3) 1–1 (4)" becomes "1–1".
def clean_score(score):
    score = str(score)
    score = re.sub(r"\(\d+\)\s*", "", score)
    return score.strip()


# Apply the cleaning functions to both teams and the score.
df["Home"] = df["Home"].apply(clean_team_name)
df["Away"] = df["Away"].apply(clean_team_name)
df["Score"] = df["Score"].apply(clean_score)

# Inspect the cleaned team names and scores.
display(df[["Home", "Score", "Away"]].head(10))

,Home,Score,Away
0,Mexico,2–0,South Africa
1,Korea Republic,2–1,Czechia
2,Canada,1–1,Bosnia–Herz
3,USA,4–1,Paraguay
4,Qatar,1–1,Switzerland
5,Brazil,1–1,Morocco
6,Australia,2–0,Türkiye
7,Haiti,0–1,Scotland
8,Germany,7–1,Curaçao
9,Netherlands,2–2,Japan


Penalty-shootout goals are removed because they are not part of the actual match score. For example, if a knockout match finished `1–1` after extra time, both teams are recorded as scoring one goal.

## 4. Split the Score into Home and Away Goals

The `Score` column currently contains both teams’ goals as text, for example `2–0`. We split this into two numeric columns:

- `home_goals`: goals scored by the home team
- `away_goals`: goals scored by the away team

In [34]:
# Split the Score column into home_goals and away_goals

df[["home_goals", "away_goals"]] = df["Score"].str.split("–", expand=True)

# Convert both goal columns from text to numeric values

df["home_goals"] = pd.to_numeric(df["home_goals"])
df["away_goals"] = pd.to_numeric(df["away_goals"])

# Display the first five cleaned matches
df[["Home", "home_goals", "away_goals", "Away"]].head()

,Home,home_goals,away_goals,Away
0,Mexico,2,0,South Africa
1,Korea Republic,2,1,Czechia
2,Canada,1,1,Bosnia–Herz
3,USA,4,1,Paraguay
4,Qatar,1,1,Switzerland


## 5. Create the Clean Match-Level Dataset

We now create a clean dataset with one row per match. Each row will contain:

- Match ID
- Tournament stage
- Match date
- Home team
- Away team
- Home goals
- Away goals

This dataset will later be used to create 208 team-match observations for Linear Regression 2.2.

In [35]:
# Convert the Date column to datetime format

df["Date"] = pd.to_datetime(df["Date"])

# Reset the index and create a unique match ID for each match

df = df.reset_index(drop=True)
df["match_id"] = df.index + 1

# Select only the columns needed for Linear Regression 2.2

matches_clean = df[[
    "match_id", "Round", "Date", "Home", "Away",
    "home_goals", "away_goals"
]].copy()

# Rename the columns to simpler and more consistent names

matches_clean.columns = [
    "match_id", "stage", "match_date",
    "home_team", "away_team",
    "home_goals", "away_goals"
]

# Save the cleaned match-level dataset

matches_clean.to_csv(
    "../data/processed/matches_clean.csv",
    index=False
)

# Display the first five rows of the cleaned dataset
matches_clean.head()

,match_id,stage,match_date,home_team,away_team,home_goals,away_goals
0,1,Group stage,2026-06-11,Mexico,South Africa,2,0
1,2,Group stage,2026-06-11,Korea Republic,Czechia,2,1
2,3,Group stage,2026-06-12,Canada,Bosnia–Herz,1,1
3,4,Group stage,2026-06-12,USA,Paraguay,4,1
4,5,Group stage,2026-06-13,Qatar,Switzerland,1,1


In [36]:
# Check the final cleaned dataset

print("Number of matches:", matches_clean["match_id"].nunique())
print("Number of rows:", len(matches_clean))
print("Missing home or away goals:")
print(matches_clean[["home_goals", "away_goals"]].isna().sum())

Number of matches: 104
Number of rows: 104
Missing home or away goals:
home_goals    0
away_goals    0
dtype: int64


## 6. Validation Results

After removing three blank separator rows, the cleaned dataset contains
104 rows and 104 unique match IDs. Both goal columns contain zero missing
values.

The cleaned match table is ready to be converted into 208 team-match
observations: one observation for each team in each match.